# 18 — Securing MCP Servers with Microsoft Entra ID

An MCP server behind API Management has two security boundaries to think
about, drawn from
[Secure access to MCP servers](https://learn.microsoft.com/en-us/azure/api-management/secure-mcp-servers):

- **Inbound**: from the MCP client (your agent) to API Management.
- **Outbound**: from API Management to the MCP server's backend (the
  underlying REST API or existing MCP server).

You can secure either, both, or neither -- but for anything beyond a local
demo, secure both.

## Inbound option 1: subscription keys

The simplest inbound control reuses API Management's existing
subscription-key mechanism: associate the MCP server with a **product**
that requires a subscription, and clients present the key in the
`Ocp-Apim-Subscription-Key` header (exactly what
`src/apim_mcp_client.py`'s `subscription_key` parameter sets -- you already
used this in notebook 17).

This is good for internal/trusted clients and quick demos, but a
subscription key is a shared secret with no per-user identity attached --
use Entra ID tokens (below) whenever you need to know *who* is calling.

## Inbound option 2: OAuth 2.1 / Microsoft Entra ID tokens

MCP clients can instead present a Microsoft Entra ID access token in the
`Authorization` header. Validate it with the
[`validate-azure-ad-token`](https://learn.microsoft.com/en-us/azure/api-management/validate-azure-ad-token-policy)
policy on the MCP server:

```xml
<inbound>
    <base />
    <validate-azure-ad-token
        tenant-id="{{entra-tenant-id}}"
        header-name="Authorization"
        failed-validation-httpcode="401"
        failed-validation-error-message="Unauthorized. Access token is missing or invalid.">
        <client-application-ids>
            <application-id>{{mcp-server-app-id}}</application-id>
        </client-application-ids>
    </validate-azure-ad-token>
</inbound>
```

Add this in the MCP server's **Policies** blade (notebook 19 covers that
blade in depth). `{{entra-tenant-id}}` and `{{mcp-server-app-id}}` are
[named values](https://learn.microsoft.com/en-us/azure/api-management/api-management-howto-properties)
-- don't hardcode tenant/app IDs directly in policy XML you might share.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.apim_mcp_client import connect_apim_mcp, get_entra_bearer_token
from src.config import require_apim_mcp

settings = require_apim_mcp()
if not settings.apim_mcp_scope:
    raise RuntimeError(
        "Set APIM_MCP_SCOPE in your .env (e.g. api://<app-id>/.default) to "
        "run this cell against a token-protected MCP server."
    )

token = get_entra_bearer_token(settings.apim_mcp_scope)

async def list_tools_with_entra_token():
    async with connect_apim_mcp(settings.apim_mcp_server_url, bearer_token=token) as session:
        return await session.list_tools()

result = await list_tools_with_entra_token()
print(f"Authenticated as an Entra ID identity; server exposes {len(result.tools)} tool(s).")